# 01c — Générer les CSV métriques à partir des traces Mirabelle

Ce notebook exécute les scripts Python présents dans `scripts_Mirabelle/` afin
de produire les fichiers de métriques par étudiant à partir des traces
Mirabelle 

## 1. Configuration

Les paramètres ci-dessous peuvent être modifiés avant l'exécution.

In [ ]:
from pathlib import Path
import sys
import shutil
import os
import runpy
import contextlib
import traceback
import pandas as pd
from IPython.display import display

def detect_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Chaine",
        Path.cwd() / "chaine_extracted" / "Chaine",
        Path.cwd().parent / "Chaine",
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "scripts_Mirabelle").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Impossible de détecter automatiquement le dossier Chaine. "
        "Modifiez PROJECT_DIR manuellement dans cette cellule."
    )

# PROJECT_DIR = Path(r"C:/Users/.../Chaine")
PROJECT_DIR = detect_project_dir()

DATA_DIR = PROJECT_DIR / "data"
SCRIPTS_DIR = PROJECT_DIR / "scripts_Mirabelle"
CSV_DIR = PROJECT_DIR / "csv"
LOG_DIR = CSV_DIR / "logs"

# Fichier de traces à traiter.
#TRACE_FILE = DATA_DIR / "traces_anonymisees_RGPD_2026_06_09.csv"
TRACE_FILE = DATA_DIR / "traces_20_sept_10_11.csv"

# Paramètre de tolérance (en minutes) transmis à session_count_Mirabelle.py pour
# regrouper les Run.Test en sessions.
GAP_MINUTES = 5.0

# Si True, supprime uniquement les CSV produits par les scripts listés
# ci-dessous avant de les régénérer.
OVERWRITE_OUTPUTS = True

CSV_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Projet :", PROJECT_DIR)
print("Scripts :", SCRIPTS_DIR)
print("Sorties CSV :", CSV_DIR)
print("Fichier de traces :", TRACE_FILE)


Projet : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine
Scripts : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\scripts_Mirabelle
Sorties CSV : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\csv
Fichier de traces : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\data\traces_anonymisees_RGPD_2026_06_09.csv


## 2. Vérification des fichiers d'entrée

On charge chaque fichier de traces pour vérifier sa forme et la présence des
colonnes attendues par les scripts (`actor`, `verb`, `tests`,
`timestamp.$date`), ainsi que des colonnes optionnelles utilisées pour la
segmentation fine des tentatives (`filename_infere`, `session.id`,
`P_codeState`, voir `verdict_utils.get_segments_indexes`).

In [12]:
if not SCRIPTS_DIR.exists():
    raise FileNotFoundError(f"Dossier scripts_Mirabelle introuvable : {SCRIPTS_DIR}")

if not TRACE_FILE.exists():
    raise FileNotFoundError(f"Fichier de traces introuvable : {TRACE_FILE}")

CORE_COLUMNS = ["actor", "verb", "tests", "timestamp.$date"]
OPTIONAL_SEGMENTATION_COLUMNS = ["filename_infere", "session.id", "P_codeState"]

trace_df = pd.read_csv(TRACE_FILE, on_bad_lines="skip", engine="python")

missing_core = [c for c in CORE_COLUMNS if c not in trace_df.columns]
missing_optional = [c for c in OPTIONAL_SEGMENTATION_COLUMNS if c not in trace_df.columns]

overview_df = pd.DataFrame([{
    "fichier": TRACE_FILE.name,
    "lignes": trace_df.shape[0],
    "colonnes": trace_df.shape[1],
    "acteurs_uniques": trace_df["actor"].nunique() if "actor" in trace_df.columns else None,
    "colonnes_requises_manquantes": ", ".join(missing_core) if missing_core else "",
    "colonnes_optionnelles_manquantes": ", ".join(missing_optional) if missing_optional else "",
}])

display(overview_df)


,fichier,lignes,colonnes,acteurs_uniques,colonnes_requises_manquantes,colonnes_optionnelles_manquantes
0,traces_anonymisees_RGPD_2026_06_09.csv,766162,34,232,,


## 3. Définition des scripts à exécuter

In [ ]:
SCRIPT_JOBS = [
    {
        "script": "session_count_Mirabelle.py",
        "output": "session_count.csv",
        "description": "Nombre de sessions de Run.Test par étudiant",
        "required": ["actor", "verb", "timestamp.$date"],
        "extra_args": [str(GAP_MINUTES)],
    },
    {
        "script": "session_span_Mirabelle.py",
        "output": "session_span.csv",
        "description": "Durée (min) entre le premier et le dernier Run.Test",
        "required": ["actor", "verb", "timestamp.$date"],
    },
    {
        "script": "failed_run_ratio_Mirabelle.py",
        "output": "failed_run_ratio.csv",
        "description": "Ratio de tirs de tests non vides contenant au moins un FailedVerdict",
        "required": ["actor", "verb", "tests"],
    },
    {
        "script": "exception_run_ratio_Mirabelle.py",
        "output": "exception_run_ratio.csv",
        "description": "Ratio de tirs de tests non vides contenant au moins un ExceptionVerdict",
        "required": ["actor", "verb", "tests"],
    },
    {
        "script": "testratio_Mirabelle.py",
        "output": "testratio.csv",
        "description": "Taux de cas de test passés (Run.Test)",
        "required": ["actor", "verb", "tests"],
    },
    {
        "script": "eq_FE_Mirabelle.py",
        "output": "eq_FE.csv",
        "description": "Error Quotient de Jadud (granularité verdict)",
        "required": ["actor", "verb", "tests", "timestamp.$date"],
    },
    {
        "script": "eq_Mirabelle.py",
        "output": "eq.csv",
        "description": "Error Quotient de Jadud (granularité message d'erreur)",
        "required": ["actor", "verb", "tests", "timestamp.$date"],
    },
    {
        "script": "red_FE_Mirabelle.py",
        "output": "RED_FE.csv",
        "description": "Repeated Error Density (granularité verdict)",
        "required": ["actor", "verb", "tests", "timestamp.$date"],
    },
    {
        "script": "red_Mirabelle.py",
        "output": "RED.csv",
        "description": "Repeated Error Density (granularité message d'erreur)",
        "required": ["actor", "verb", "tests", "timestamp.$date"],
    },
]

jobs_df = pd.DataFrame([
    {k: v for k, v in job.items() if k not in {"required", "extra_args"}}
    for job in SCRIPT_JOBS
])
display(jobs_df)


,script,output,description
0,session_count_Mirabelle.py,session_count.csv,Nombre de sessions de Run.Test par étudiant
1,session_span_Mirabelle.py,session_span.csv,Durée (min) entre le premier et le dernier Run...
2,testratio_Mirabelle.py,testratio.csv,Taux de cas de test passés (Run.Test)
3,eq_FE_Mirabelle.py,eq_FE.csv,Error Quotient de Jadud (granularité verdict)
4,eq_Mirabelle.py,eq.csv,Error Quotient de Jadud (granularité message d...
5,red_FE_Mirabelle.py,RED_FE.csv,Repeated Error Density (granularité verdict)
6,red_Mirabelle.py,RED.csv,Repeated Error Density (granularité message d'...


## 4. Exécution des scripts

Chaque script est exécuté une fois par fichier de traces, avec en
argument le chemin du fichier de traces et le chemin de sortie du CSV
attendu (voir la partie `if __name__ == "__main__":` de chaque script).

In [14]:
def missing_columns(job, columns):
    return [c for c in job.get("required", []) if c not in columns]

def output_summary(path: Path):
    if not path.exists():
        return {"rows": None, "columns": None}
    try:
        df = pd.read_csv(path)
        return {
            "rows": int(df.shape[0]),
            "columns": ", ".join(df.columns.astype(str).tolist()),
        }
    except Exception as exc:
        return {"rows": None, "columns": f"lecture impossible : {exc}"}

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

report = []
trace_columns = set(trace_df.columns)

for job in SCRIPT_JOBS:
    print(f"Lancement : {job['script']} → {job['output']}", flush=True)

    script_path = SCRIPTS_DIR / job["script"]
    output_path = CSV_DIR / job["output"]
    log_path = LOG_DIR / f"{Path(job['script']).stem}.log"

    row = {
        "script": job["script"],
        "output": job["output"],
        "description": job["description"],
        "status": None,
        "returncode": None,
        "output_path": str(output_path),
        "rows": None,
        "columns": None,
        "log_path": str(log_path),
        "message": "",
    }

    if not script_path.exists():
        row.update(status="skipped", message=f"Script introuvable : {script_path}")
        report.append(row)
        print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
        continue

    missing = missing_columns(job, trace_columns)
    if missing:
        row.update(status="skipped_missing_columns", message=f"Colonnes manquantes : {missing}")
        report.append(row)
        print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
        continue

    if OVERWRITE_OUTPUTS and output_path.exists():
        output_path.unlink()

    argv = [
        str(script_path),
        str(TRACE_FILE),
        str(output_path),
        *job.get("extra_args", []),
    ]

    old_argv = sys.argv[:]
    try:
        with log_path.open("w", encoding="utf-8") as log_file:
            log_file.write("COMMANDE LOGIQUE\npython " + " ".join(argv) + "\n\nSORTIE DU SCRIPT\n")
            log_file.flush()
            with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                sys.argv = argv
                try:
                    runpy.run_path(str(script_path), run_name="__main__")
                    returncode = 0
                except SystemExit as exc:
                    returncode = int(exc.code or 0) if isinstance(exc.code, int) else 1
        summary = output_summary(output_path)
        status = "ok" if returncode == 0 and output_path.exists() else "error"
        message = "CSV produit" if output_path.exists() else "Aucun CSV produit"
    except Exception as exc:
        returncode = 1
        status = "error"
        message = f"{type(exc).__name__}: {exc}"
        with log_path.open("a", encoding="utf-8") as log_file:
            log_file.write("\n\nEXCEPTION NOTEBOOK\n")
            traceback.print_exc(file=log_file)
        summary = output_summary(output_path)
    finally:
        sys.argv = old_argv

    row.update(
        status=status,
        returncode=returncode,
        rows=summary["rows"],
        columns=summary["columns"],
        message=message,
    )
    print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
    report.append(row)

report_df = pd.DataFrame(report)
report_path = LOG_DIR / "run_report.csv"
report_df.to_csv(report_path, index=False)

display(report_df)
print(f"Rapport sauvegardé : {report_path}")


Lancement : session_count_Mirabelle.py → session_count.csv
Terminé : session_count_Mirabelle.py — ok (CSV produit)
Lancement : session_span_Mirabelle.py → session_span.csv
Terminé : session_span_Mirabelle.py — ok (CSV produit)
Lancement : testratio_Mirabelle.py → testratio.csv
Terminé : testratio_Mirabelle.py — ok (CSV produit)
Lancement : eq_FE_Mirabelle.py → eq_FE.csv
Terminé : eq_FE_Mirabelle.py — ok (CSV produit)
Lancement : eq_Mirabelle.py → eq.csv
Terminé : eq_Mirabelle.py — ok (CSV produit)
Lancement : red_FE_Mirabelle.py → RED_FE.csv
Terminé : red_FE_Mirabelle.py — ok (CSV produit)
Lancement : red_Mirabelle.py → RED.csv
Terminé : red_Mirabelle.py — ok (CSV produit)


,script,output,description,status,returncode,output_path,rows,columns,log_path,message
0,session_count_Mirabelle.py,session_count.csv,Nombre de sessions de Run.Test par étudiant,ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,208,"SubjectID, SessionCount",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
1,session_span_Mirabelle.py,session_span.csv,Durée (min) entre le premier et le dernier Run...,ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,208,"SubjectID, SessionSpan",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
2,testratio_Mirabelle.py,testratio.csv,Taux de cas de test passés (Run.Test),ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,204,"SubjectID, TestPassRate",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
3,eq_FE_Mirabelle.py,eq_FE.csv,Error Quotient de Jadud (granularité verdict),ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,204,"SubjectID, ErrorQuotient_FE",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
4,eq_Mirabelle.py,eq.csv,Error Quotient de Jadud (granularité message d...,ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,204,"SubjectID, ErrorQuotient",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
5,red_FE_Mirabelle.py,RED_FE.csv,Repeated Error Density (granularité verdict),ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,204,"SubjectID, RED_FE",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
6,red_Mirabelle.py,RED.csv,Repeated Error Density (granularité message d'...,ok,0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,204,"SubjectID, RED",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit


Rapport sauvegardé : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\csv\logs\run_report.csv


## 5. Contrôle des sorties

In [15]:
generated = []
for csv_path in sorted(CSV_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
        generated.append({
            "fichier": csv_path.name,
            "lignes": df.shape[0],
            "colonnes": df.shape[1],
            "noms_colonnes": ", ".join(df.columns.astype(str).tolist()),
        })
    except Exception as exc:
        generated.append({
            "fichier": csv_path.name,
            "lignes": None,
            "colonnes": None,
            "noms_colonnes": f"lecture impossible : {exc}",
        })

generated_df = pd.DataFrame(generated)
display(generated_df)


,fichier,lignes,colonnes,noms_colonnes
0,eq.csv,204,2,"SubjectID, ErrorQuotient"
1,eq_FE.csv,204,2,"SubjectID, ErrorQuotient_FE"
2,RED.csv,204,2,"SubjectID, RED"
3,RED_FE.csv,204,2,"SubjectID, RED_FE"
4,session_count.csv,208,2,"SubjectID, SessionCount"
5,session_span.csv,208,2,"SubjectID, SessionSpan"
6,testratio.csv,204,2,"SubjectID, TestPassRate"


## 6. Affichage des journaux des erreurs éventuelles

In [16]:
error_rows = report_df[report_df["status"].isin(["error", "skipped_missing_columns", "skipped"])].copy()

if error_rows.empty:
    print("Aucune erreur détectée.")
else:
    display(error_rows[["script", "status", "message", "log_path"]])
    for _, row in error_rows.iterrows():
        log_path = Path(row["log_path"])
        print("\n" + "=" * 100)
        print(f"{row['script']} — {row['status']}")
        print(row["message"])
        if log_path.exists():
            log_text = log_path.read_text(encoding="utf-8", errors="replace")
            print(log_text[-4000:])


Aucune erreur détectée.
